# Magnetic Field vs. Position and Current — On-Axis Field of a Multi-Turn Coil

**TODO before sending:** fill in your actual measurement setup in 1-2 sentences (rail length,
probe type, how currents/positions were chosen) -- I only know it's a multi-turn coil, not the
physical details of how you measured it.

**Setup:** a multi-turn coil, current I, field measured along its axis at position x with a
field probe.

**Goal:** find a model for B(x, I) and check it against theory, rather than just curve-fitting
for its own sake.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

DATA_PATH = Path("magnetic_field_experiment.csv")   # CSV must sit next to this notebook
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.head()


## 1. Data quality

238 rows, 13 positions x 7 currents x 3 trials.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nRows sharing a timestamp:\n{df[df['timestamp'].duplicated(keep=False)]}")


## 2. Cleaning

Two corrections, both logged with a reason rather than applied silently.

In [ ]:
df_clean = df.copy()
correction_log = pd.DataFrame(columns=[
    "measurement_id", "original_position", "corrected_position", "reason"
])

def log_correction(measurement_id, original_position, corrected_position, reason):
    global correction_log
    correction_log = pd.concat([correction_log, pd.DataFrame([{
        "measurement_id": measurement_id, "original_position": original_position,
        "corrected_position": corrected_position, "reason": reason,
    }])], ignore_index=True)


In [ ]:
# Measurement 232 was logged at position 6, but its field value only makes sense next to
# the position-12 cluster for this current -- shown here before the fix is applied.
check = df_clean[df_clean["current"] == 1.5].sort_values("position")
fig, ax = plt.subplots()
ax.scatter(check["position"], check["magnetic_field"], label="all current=1.5A points")
flagged = check[check["measurement_id"] == 232]
ax.scatter(flagged["position"], flagged["magnetic_field"], color="red", zorder=3,
           label="measurement 232 (as logged)")
ax.set_xlabel("Position (cm)"); ax.set_ylabel("Magnetic field (mT)")
ax.set_title("Why measurement 232 looks mislabeled")
ax.legend()
plt.show()

df_clean.loc[df_clean["measurement_id"] == 232, "position"] = 12
log_correction(232, 6, 12, "Field value falls on the position=12 trend for this current, "
                           "not position=6; logged position was almost certainly a "
                           "transcription error.")

df_clean = df_clean[df_clean["measurement_id"] != 235].copy()
log_correction(235, 12, 12, "Exact duplicate of measurement 154 (identical timestamp and "
                            "values); removed so it isn't counted twice.")

df_analysis = df_clean.dropna(subset=["magnetic_field", "field_uncertainty"]).copy()
correction_log


## 3. B(x) at fixed current, with error bars

In [ ]:
def plot_field_vs_position(data, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.errorbar(data["position"], data["magnetic_field"], yerr=data["field_uncertainty"],
                fmt="o", markersize=4, capsize=3)
    ax.set_xlabel("Position (cm)"); ax.set_ylabel("Magnetic field (mT)")
    ax.set_title(title)
    return ax

for current in sorted(df_analysis["current"].unique()):
    if current == 0:
        continue
    subset = df_analysis[df_analysis["current"] == current]
    plot_field_vs_position(subset, f"Magnetic field vs. position -- {current} A")
    plt.show()


## 4. First pass: single-loop model

B(x) is peaked and symmetric -- as a starting point, try the on-axis field of a single thin
circular loop:

$$B(x) = \frac{k I}{(1 + (x/a)^2)^{3/2}}$$

Fit weighted by `field_uncertainty`, with `absolute_sigma=True` so the reported errors and
chi-squared are physically meaningful.

In [ ]:
def loop_field_global(X, k, a):
    x, I = X
    return k * I / (1 + (x / a) ** 2) ** 1.5

fit_data = df_analysis[df_analysis["current"] > 0]
popt, pcov = curve_fit(loop_field_global, (fit_data["position"], fit_data["current"]),
                        fit_data["magnetic_field"], p0=[2.4, 9.0],
                        sigma=fit_data["field_uncertainty"], absolute_sigma=True)
k_fit, a_fit = popt
k_err, a_err = np.sqrt(np.diag(pcov))
predicted = loop_field_global((fit_data["position"], fit_data["current"]), *popt)
resid = fit_data["magnetic_field"] - predicted
chi_sq = np.sum((resid / fit_data["field_uncertainty"]) ** 2)
dof = len(fit_data) - 2

print(f"k = {k_fit:.4f} +/- {k_err:.4f}  mT/A")
print(f"a = {a_fit:.3f} +/- {a_err:.3f}  cm")
print(f"reduced chi-squared = {chi_sq/dof:.1f}  (n = {len(fit_data)}, dof = {dof})")


Reduced chi-squared here is large (~55), far from the ~1 a good fit should give. Before
concluding anything, check whether that's because the measurement uncertainty is
underestimated, or because the model itself is wrong.

In [ ]:
trial_std = df_analysis.groupby(["position", "current"])["magnetic_field"].std()
print(f"Mean trial-to-trial std dev:      {trial_std.mean():.4f} mT")
print(f"Mean reported field_uncertainty:  {df_analysis['field_uncertainty'].mean():.4f} mT")
print(f"Mean |fit residual|:              {resid.abs().mean():.4f} mT")
print(f"Correlation of residual with |position|: "
      f"{np.corrcoef(fit_data['position'].abs(), resid / fit_data['field_uncertainty'])[0,1]:.2f}")


The trial-to-trial scatter matches the reported `field_uncertainty` almost exactly, so the
uncertainty isn't underestimated. But the residuals are ~6x larger than that repeat-measurement
noise, with a mild negative correlation to position -- pointing at the model, not the data.
That fits: **this is a multi-turn coil, not a single thin loop**, so the single-loop formula is
only an approximation.

## 5. Second pass: finite-solenoid model

For a coil of N turns wound uniformly over a length L (turn density n = N/L), the on-axis field
is the superposition of every turn's loop field, which integrates to:

$$B(x) = \frac{k I}{2}\left[\frac{x + L/2}{\sqrt{a^2 + (x+L/2)^2}} - \frac{x - L/2}{\sqrt{a^2 + (x-L/2)^2}}\right]$$

where `a` is the coil radius and `k` absorbs $\mu_0 n$. This has three physical parameters
(k, a, L) instead of two, but each has a direct physical meaning, and it reduces to the
single-loop formula as L -> 0.

In [ ]:
def solenoid_field(X, k, a, L):
    x, I = X
    term1 = (x + L / 2) / np.sqrt(a ** 2 + (x + L / 2) ** 2)
    term2 = (x - L / 2) / np.sqrt(a ** 2 + (x - L / 2) ** 2)
    return k * I * (term1 - term2)

popt2, pcov2 = curve_fit(
    solenoid_field, (df_analysis["position"], df_analysis["current"]), df_analysis["magnetic_field"],
    p0=[1.2, 9.0, 5.0], sigma=df_analysis["field_uncertainty"], absolute_sigma=True, maxfev=10000,
)
k2, a2, L2 = popt2
k2_err, a2_err, L2_err = np.sqrt(np.diag(pcov2))
predicted2 = solenoid_field((df_analysis["position"], df_analysis["current"]), *popt2)
resid2 = df_analysis["magnetic_field"] - predicted2
chi_sq2 = np.sum((resid2 / df_analysis["field_uncertainty"]) ** 2)
dof2 = len(df_analysis) - 3

print(f"k = {k2:.4f} +/- {k2_err:.4f}  mT/A")
print(f"a (radius) = {a2:.3f} +/- {a2_err:.3f}  cm")
print(f"L (winding length) = {L2:.3f} +/- {L2_err:.3f}  cm")
print(f"reduced chi-squared = {chi_sq2/dof2:.2f}  (n = {len(df_analysis)}, dof = {dof2})")


Reduced chi-squared drops from ~55 to about 1.3 -- essentially consistent with the
measurement uncertainty -- and the residual-vs-position correlation drops to nearly zero. The
finite-solenoid model is the right one for this coil; the single-loop model in Section 4 was a
reasonable first attempt but genuinely the wrong shape for this apparatus.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
x_smooth = np.linspace(-13, 13, 200)
for ax, current in zip(axes.flat, sorted(df_analysis[df_analysis["current"] > 0]["current"].unique())):
    subset = df_analysis[df_analysis["current"] == current]
    ax.errorbar(subset["position"], subset["magnetic_field"], yerr=subset["field_uncertainty"],
                fmt="o", markersize=3, capsize=2, label="data")
    ax.plot(x_smooth, solenoid_field((x_smooth, np.full_like(x_smooth, current)), *popt2),
            color="tab:red", label="solenoid fit")
    ax.set_title(f"{current} A")
axes[0, 0].legend(fontsize=8)
fig.supxlabel("Position (cm)"); fig.supylabel("Magnetic field (mT)")
plt.tight_layout()
plt.show()


## 6. Symmetry check

Field at $+x$ vs. $-x$ compared directly, with uncertainty propagated on the difference.

In [ ]:
def symmetry_check(subset):
    pos = subset[subset["position"] > 0].set_index("position")
    neg = subset[subset["position"] < 0].copy()
    neg["position"] = -neg["position"]
    neg = neg.set_index("position")
    paired = pos.join(neg, lsuffix="_pos", rsuffix="_neg", how="inner")
    paired["difference"] = paired["magnetic_field_pos"] - paired["magnetic_field_neg"]
    paired["difference_err"] = np.sqrt(paired["field_uncertainty_pos"] ** 2 +
                                        paired["field_uncertainty_neg"] ** 2)
    paired["within_2sigma"] = paired["difference"].abs() <= 2 * paired["difference_err"]
    return paired[["magnetic_field_pos", "magnetic_field_neg", "difference",
                    "difference_err", "within_2sigma"]]

symmetry_check(df_analysis[df_analysis["current"] == 1.5])


## 7. Summary

- 238 measurements, 13 positions x 7 currents x 3 trials; 2 corrections logged, 1 missing value dropped.
- A single-loop model fit the overall shape of B(x) but left reduced chi-squared ~55; repeat-trial scatter matched the reported uncertainty almost exactly, ruling out uncertainty underestimation and pointing at the model.
- Since the coil is multi-turn (wound over a length, not a single point), the finite-solenoid on-axis field was fit instead: reduced chi-squared drops to ~1.3, and residuals lose their positional trend.
- Fitted coil geometry: radius a = 5.49 +/- 0.02 cm, winding length L = 14.01 +/- 0.04 cm.
- Left/right symmetry about position 0 holds within propagated uncertainty.

**Next steps:** compare the fitted radius and winding length to a direct physical measurement
of the coil, if available -- that would be the strongest possible check of this result.
